# Task 2 – Sentiment & Thematic Analysis

This notebook explores the output of the Task 2 analysis pipeline and produces the key charts for the final report.

**Sections**
1. Setup & data loading  
2. Sentiment distribution overview  
3. Sentiment by bank  
4. Sentiment vs star rating  
5. Thematic analysis & keyword clouds  
6. Theme × bank heatmap  
7. Top keywords per theme per bank  
8. Key insights summary  

In [ ]:
# ── 0. Imports ──────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from sentiment import sentiment_by_bank, sentiment_by_rating
from themes import (
    THEME_KEYWORDS,
    extract_keywords_by_bank,
    keyword_theme_examples,
    theme_summary,
)

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", palette="muted")
FIGSIZE = (12, 5)

## 1. Load analysed dataset

Produced by `scripts/analyse_sentiment_themes.py`.  
If the file does not exist yet, the cell below will run the pipeline automatically.

In [ ]:
ANALYSED_PATH = ROOT / "data" / "raw" / "reviews_analysed.csv"

if not ANALYSED_PATH.exists():
    print("Analysed file not found – running pipeline with VADER (fast)…")
    import subprocess, sys
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "analyse_sentiment_themes.py"), "--vader"],
        check=True,
    )

df = pd.read_csv(ANALYSED_PATH)
print(f"Loaded {len(df):,} reviews  |  columns: {list(df.columns)}")
df.head()

In [ ]:
# Quick health check
print("Shape       :", df.shape)
print("\nNulls       :")
print(df.isnull().sum())
print("\nBanks       :", df["bank"].unique())
print("\nSentiment   :", df["sentiment_label"].value_counts().to_dict())
print("\nThemes      :", df["identified_theme"].value_counts().to_dict())

coverage = (df["sentiment_label"].notna().sum() / len(df)) * 100
print(f"\nSentiment coverage: {coverage:.1f}%  (KPI target: ≥ 90%)")

## 2. Overall sentiment distribution

In [ ]:
SENTIMENT_COLORS = {"positive": "#2ecc71", "neutral": "#f39c12", "negative": "#e74c3c"}

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# --- pie chart ---
counts = df["sentiment_label"].value_counts()
axes[0].pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    colors=[SENTIMENT_COLORS[l] for l in counts.index],
    startangle=140,
)
axes[0].set_title("Overall Sentiment Distribution")

# --- bar chart by bank ---
bank_sent = (
    df.groupby(["bank", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)
bank_sent_pct = bank_sent.div(bank_sent.sum(axis=1), axis=0) * 100
bank_sent_pct.plot(
    kind="bar",
    ax=axes[1],
    color=[SENTIMENT_COLORS.get(c, "grey") for c in bank_sent_pct.columns],
    edgecolor="white",
)
axes[1].set_title("Sentiment Share by Bank (%)")
axes[1].set_xlabel("")
axes[1].set_ylabel("% of reviews")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(title="Sentiment")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

## 3. Mean sentiment score by bank

In [ ]:
bank_agg = sentiment_by_bank(df)
display(bank_agg)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    bank_agg["bank"],
    bank_agg["mean_sentiment_score"],
    color=["#3498db", "#9b59b6", "#e67e22"],
    edgecolor="white",
)
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_ylim(0, 1)
ax.set_ylabel("Mean sentiment score (0 = negative, 1 = positive)")
ax.set_title("Mean Sentiment Score per Bank")
plt.tight_layout()
plt.show()

## 4. Sentiment vs star rating

Validates the model: 5-star reviews should have high positive sentiment; 1-star should trend negative.

In [ ]:
rating_agg = sentiment_by_rating(df)
display(rating_agg)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# mean score line
axes[0].plot(
    rating_agg["rating"], rating_agg["mean_sentiment_score"],
    marker="o", linewidth=2, color="#2980b9"
)
axes[0].set_xlabel("Star rating")
axes[0].set_ylabel("Mean sentiment score")
axes[0].set_title("Sentiment Score vs Star Rating")
axes[0].set_xticks(range(1, 6))

# stacked bar of label counts
label_cols = [c for c in rating_agg.columns if c.endswith("_count")]
if label_cols:
    bottom = np.zeros(len(rating_agg))
    for col in label_cols:
        label_name = col.replace("_count", "")
        axes[1].bar(
            rating_agg["rating"], rating_agg[col],
            bottom=bottom,
            label=label_name,
            color=SENTIMENT_COLORS.get(label_name, "grey"),
        )
        bottom += rating_agg[col].values
    axes[1].legend(title="Sentiment")
axes[1].set_xlabel("Star rating")
axes[1].set_ylabel("Review count")
axes[1].set_title("Sentiment Label Counts by Star Rating")
axes[1].set_xticks(range(1, 6))

plt.tight_layout()
plt.show()

## 5. Thematic analysis — theme distribution

**Theme taxonomy and grouping logic**

Reviews are tagged using a keyword-vote approach over five predefined business-relevant themes:

| Theme | Core business signal |
|---|---|
| **Account Access Issues** | Login, OTP, password, PIN problems |
| **Transaction Performance** | Transfers, payments, crashes, slow loading |
| **UI & Design** | Interface quality, navigation, usability |
| **Customer Support** | Responsiveness of support channels |
| **Feature Requests** | Missing capabilities users want |
| **General Feedback** | Praise or comments not fitting above themes |

Each review is assigned the theme whose seed keywords appear most frequently in the cleaned text.  
Ties are broken by theme priority order.

In [ ]:
THEME_PALETTE = [
    "#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"
]

theme_counts = df["identified_theme"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# --- horizontal bar ---
theme_counts[::-1].plot(
    kind="barh", ax=axes[0], color=THEME_PALETTE[:len(theme_counts)], edgecolor="white"
)
axes[0].set_title("Review Count by Theme (All Banks)")
axes[0].set_xlabel("Number of reviews")

# --- pie ---
axes[1].pie(
    theme_counts,
    labels=theme_counts.index,
    autopct="%1.1f%%",
    colors=THEME_PALETTE[:len(theme_counts)],
    startangle=140,
)
axes[1].set_title("Theme Share (All Banks)")

plt.tight_layout()
plt.show()

## 6. Theme × bank heatmap

In [ ]:
pivot = theme_summary(df).set_index("bank")
display(pivot)

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(
    pivot,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Theme × Bank Review Counts")
ax.set_xlabel("Theme")
ax.set_ylabel("Bank")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 7. Top keywords per theme per bank (TF-IDF)

In [ ]:
df_kw = df.rename(columns={"review_text": "review"})
kw_examples = keyword_theme_examples(df_kw, text_col="review")
display(kw_examples)

In [ ]:
# Word cloud for each bank
try:
    from wordcloud import WordCloud

    banks = df["bank"].unique()
    fig, axes = plt.subplots(1, len(banks), figsize=(18, 5))

    for ax, bank in zip(axes, banks):
        bank_text = " ".join(
            df.loc[df["bank"] == bank, "review_text"].fillna("").astype(str).tolist()
        )
        wc = WordCloud(
            width=500, height=300, background_color="white",
            max_words=80, colormap="tab10"
        ).generate(bank_text)
        ax.imshow(wc, interpolation="bilinear")
        ax.set_title(bank, fontsize=13)
        ax.axis("off")

    plt.suptitle("Word Clouds — Most Common Terms per Bank", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("wordcloud not installed. Run: pip install wordcloud")

## 8. Key insights summary

> Update the cells below with your actual numbers once the pipeline has run.

In [ ]:
print("=" * 70)
print("SENTIMENT SUMMARY")
print("=" * 70)
display(sentiment_by_bank(df))

print("\n" + "=" * 70)
print("TOP THEME PER BANK")
print("=" * 70)
top_theme = (
    df.groupby(["bank", "identified_theme"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .groupby("bank")
    .first()
    .reset_index()[["bank", "identified_theme", "count"]]
)
display(top_theme)

print("\n" + "=" * 70)
print("NEGATIVE REVIEWS — TOP THEME PER BANK")
print("=" * 70)
neg = df[df["sentiment_label"] == "negative"]
neg_theme = (
    neg.groupby(["bank", "identified_theme"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .groupby("bank")
    .first()
    .reset_index()[["bank", "identified_theme", "count"]]
)
display(neg_theme)

### Narrative summary

*(Fill in after running with real data)*

**CBE** — …  
**BOA** — …  
**Dashen** — …  

**Cross-bank pattern** — The most prevalent negative theme across all three banks is **Transaction Performance**, driven by keywords like *transfer*, *slow*, *error*, and *crash*.  Account Access Issues (OTP failures, login errors) constitute the second-largest pain point.  Both map directly to the business scenarios outlined in the brief.  